# Semana 12: Sistemas de Recomendación
## Notebook de Ejercicios (NB2) – Recomendación de Películas (MovieLens)

**Propósito:** Implementar filtrado colaborativo basado en ítems y factorización matricial (SVD) para recomendar películas, evaluar con métricas de rating y simular el problema de cold start.

**Docente:** Carlos César Sánchez Coronel

**Objetivos de aprendizaje:**
- Explorar el dataset MovieLens y comprender su estructura.
- Implementar filtrado colaborativo basado en ítems usando similitud de coseno.
- Entrenar SVD con la librería Surprise y evaluar con RMSE en validación cruzada.
- Simular un escenario de cold start para nuevos usuarios.

---

## 0. Configuración Inicial

Importamos las librerías necesarias y fijamos la semilla para reproducibilidad.

In [ ]:
# Importamos librerías
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Surprise para sistemas de recomendación
try:
    from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
    from surprise.model_selection import train_test_split, cross_validate, GridSearchCV
    from surprise import dump
    SURPRISE_AVAILABLE = True
except ImportError:
    SURPRISE_AVAILABLE = False
    print("Nota: Surprise no está instalado. Algunas celdas no se ejecutarán.")
    print("Puedes instalarlo con: !pip install scikit-surprise")

# Scikit-learn para similitud de coseno
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

# Configuración de visualización
%matplotlib inline
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# Semilla
np.random.seed(42)

print("Librerías importadas correctamente.")

---
## 1. Carga y Exploración del Dataset MovieLens 100k

El dataset **MovieLens 100k** contiene 100,000 valoraciones (1-5) de 943 usuarios sobre 1,682 películas. Cada usuario ha valorado al menos 20 películas.

In [ ]:
if SURPRISE_AVAILABLE:
    # Cargamos el dataset incluido en Surprise
    data = Dataset.load_builtin('ml-100k')
    
    # Para exploración, convertimos a DataFrame
    reader = Reader(line_format='user item rating timestamp', sep='\t')
    df_ratings = pd.read_csv('http://files.grouplens.org/datasets/movielens/ml-100k/u.data', 
                              sep='\t', names=['user_id', 'item_id', 'rating', 'timestamp'])
    
    # Cargamos información de películas
    df_movies = pd.read_csv('http://files.grouplens.org/datasets/movielens/ml-100k/u.item', 
                             sep='|', encoding='latin-1', 
                             names=['movie_id', 'title', 'release_date', 'video_release_date', 
                                    'imdb_url', 'unknown', 'action', 'adventure', 'animation',
                                    'childrens', 'comedy', 'crime', 'documentary', 'drama', 
                                    'fantasy', 'film_noir', 'horror', 'musical', 'mystery', 
                                    'romance', 'sci_fi', 'thriller', 'war', 'western'])
    
    print("Dataset MovieLens 100k cargado correctamente.")
    print(f"Ratings: {df_ratings.shape}")
    print(f"Películas: {df_movies.shape}")
else:
    print("Surprise no está instalado. Generando datos sintéticos...")
    # Datos sintéticos para que el notebook no falle
    np.random.seed(42)
    n_users = 100
    n_items = 50
    n_ratings = 1000
    
    df_ratings = pd.DataFrame({
        'user_id': np.random.randint(1, n_users+1, n_ratings),
        'item_id': np.random.randint(1, n_items+1, n_ratings),
        'rating': np.random.randint(1, 6, n_ratings),
        'timestamp': np.random.randint(1000000000, 1000000000 + n_ratings, n_ratings)
    })
    
    df_movies = pd.DataFrame({
        'movie_id': range(1, n_items+1),
        'title': [f'Movie {i}' for i in range(1, n_items+1)],
        'genres': np.random.choice(['Action', 'Comedy', 'Drama', 'Horror', 'Sci-Fi'], n_items)
    })
    print("Datos sintéticos generados.")

### 1.1. Exploración de Ratings

In [ ]:
# Estadísticas básicas
print("=== Estadísticas de Ratings ===")
print(df_ratings['rating'].describe())

# Distribución de ratings
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df_ratings['rating'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribución de Ratings')
plt.xlabel('Rating')
plt.ylabel('Frecuencia')

plt.subplot(1, 2, 2)
df_ratings['rating'].hist(bins=5, edgecolor='black')
plt.title('Histograma de Ratings')
plt.xlabel('Rating')
plt.ylabel('Frecuencia')

plt.tight_layout()
plt.show()

In [ ]:
# Número de ratings por usuario
ratings_per_user = df_ratings.groupby('user_id').size()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
ratings_per_user.hist(bins=30, edgecolor='black')
plt.title('Distribución de Ratings por Usuario')
plt.xlabel('Número de ratings')
plt.ylabel('Frecuencia')

plt.subplot(1, 2, 2)
ratings_per_user.plot(kind='box')
plt.title('Boxplot de Ratings por Usuario')
plt.ylabel('Número de ratings')

plt.tight_layout()
plt.show()

print(f"Usuario con más ratings: {ratings_per_user.max()}")
print(f"Usuario con menos ratings: {ratings_per_user.min()}")

In [ ]:
# Número de ratings por película
ratings_per_movie = df_ratings.groupby('item_id').size()

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
ratings_per_movie.hist(bins=30, edgecolor='black')
plt.title('Distribución de Ratings por Película')
plt.xlabel('Número de ratings')
plt.ylabel('Frecuencia')

plt.subplot(1, 2, 2)
ratings_per_movie.plot(kind='box')
plt.title('Boxplot de Ratings por Película')
plt.ylabel('Número de ratings')

plt.tight_layout()
plt.show()

# Películas más populares
top_movies = df_ratings.groupby('item_id')['rating'].agg(['count', 'mean']).sort_values('count', ascending=False).head(10)
top_movies = top_movies.merge(df_movies[['movie_id', 'title']], left_index=True, right_on='movie_id')
print("\nPelículas más populares (más ratings):")
print(top_movies[['title', 'count', 'mean']])

---
## 2. Filtrado Colaborativo Basado en Ítems (Item-Based Collaborative Filtering)

Construimos una matriz usuario-ítem y calculamos la similitud entre ítems usando coseno.

In [ ]:
# Creamos la matriz usuario-ítem (pivot table)
user_item_matrix = df_ratings.pivot_table(index='user_id', columns='item_id', values='rating')

# Rellenamos NaN con 0 para calcular similitud
user_item_matrix_filled = user_item_matrix.fillna(0)

print(f"Dimensiones de la matriz: {user_item_matrix.shape}")
print(f"Densidad: {user_item_matrix.count().sum() / (user_item_matrix.shape[0] * user_item_matrix.shape[1]):.2%}")

# Calculamos similitud de coseno entre ítems (columnas)
item_similarity = cosine_similarity(user_item_matrix_filled.T)
item_similarity_df = pd.DataFrame(item_similarity, 
                                   index=user_item_matrix.columns, 
                                   columns=user_item_matrix.columns)

print("\nSimilitud entre ítems (primeras 5 filas y columnas):")
print(item_similarity_df.iloc[:5, :5])

### 2.1. Función para recomendar películas a un usuario

In [ ]:
def recommend_items(user_id, user_item_matrix, item_similarity_df, n_recommendations=10):
    """
    Recomienda ítems a un usuario basado en ítems que ya ha valorado.
    """
    # Obtener ítems que el usuario ya ha valorado
    user_ratings = user_item_matrix.loc[user_id].dropna()
    
    if len(user_ratings) == 0:
        return "El usuario no tiene valoraciones."
    
    # Calcular puntuaciones para todos los ítems no valorados
    scores = {}
    for item in user_item_matrix.columns:
        if pd.isna(user_item_matrix.loc[user_id, item]):
            # Suma ponderada de similitudes con ítems valorados
            sim_sum = 0
            weighted_sum = 0
            for rated_item, rating in user_ratings.items():
                sim = item_similarity_df.loc[item, rated_item]
                weighted_sum += sim * rating
                sim_sum += abs(sim)
            if sim_sum > 0:
                scores[item] = weighted_sum / sim_sum
            else:
                scores[item] = 0
    
    # Ordenar y devolver top N
    recommended_items = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:n_recommendations]
    return recommended_items

# Probamos para un usuario (ej. usuario 1)
user_example = 1
recommendations = recommend_items(user_example, user_item_matrix, item_similarity_df, n_recommendations=10)

print(f"Recomendaciones para usuario {user_example}:")
for item_id, score in recommendations:
    movie_title = df_movies[df_movies['movie_id'] == item_id]['title'].values
    if len(movie_title) > 0:
        print(f"- {movie_title[0]} (score: {score:.3f})")
    else:
        print(f"- Ítem {item_id} (score: {score:.3f})")

### 2.2. Evaluación simple del filtrado basado en ítems

Para evaluar, podríamos dejar algunos ratings fuera y ver qué tan bien predice.

In [ ]:
# Función para predecir un rating usando item-based
def predict_rating_item_based(user_id, item_id, user_item_matrix, item_similarity_df):
    user_ratings = user_item_matrix.loc[user_id].dropna()
    if len(user_ratings) == 0:
        return np.nan
    
    sim_sum = 0
    weighted_sum = 0
    for rated_item, rating in user_ratings.items():
        sim = item_similarity_df.loc[item_id, rated_item]
        weighted_sum += sim * rating
        sim_sum += abs(sim)
    
    if sim_sum > 0:
        return weighted_sum / sim_sum
    else:
        return np.nan

# Evaluamos en un conjunto pequeño
from sklearn.metrics import mean_squared_error

# Tomamos algunos ratings conocidos para prueba
test_ratings = df_ratings.sample(100, random_state=42)
predictions = []
actuals = []

for _, row in test_ratings.iterrows():
    user_id = row['user_id']
    item_id = row['item_id']
    actual = row['rating']
    
    if user_id in user_item_matrix.index and item_id in user_item_matrix.columns:
        pred = predict_rating_item_based(user_id, item_id, user_item_matrix, item_similarity_df)
        if not np.isnan(pred):
            predictions.append(pred)
            actuals.append(actual)

if len(predictions) > 0:
    rmse = np.sqrt(mean_squared_error(actuals, predictions))
    print(f"RMSE en muestra de prueba (item-based): {rmse:.4f}")
else:
    print("No se pudieron hacer predicciones.")

---
## 3. SVD con Surprise: Entrenamiento y Evaluación

Usamos la librería Surprise para entrenar SVD y evaluar con validación cruzada.

In [ ]:
if SURPRISE_AVAILABLE:
    # Configuración de SVD
    algo = SVD(n_factors=20, reg_all=0.1, lr_all=0.005, n_epochs=20, random_state=42)
    
    # Validación cruzada
    cv_results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
    
    print("\nResultados de validación cruzada (SVD):")
    print(f"RMSE medio: {np.mean(cv_results['test_rmse']):.4f} (+/- {np.std(cv_results['test_rmse']):.4f})")
    print(f"MAE medio: {np.mean(cv_results['test_mae']):.4f} (+/- {np.std(cv_results['test_mae']):.4f})")
    
    # Entrenamos en todo el dataset para uso posterior
    trainset = data.build_full_trainset()
    algo.fit(trainset)
else:
    print("Surprise no está instalado. Omitiendo esta sección.")

---
## 4. Simulación de Cold Start

El problema de **cold start** ocurre cuando un nuevo usuario no tiene valoraciones previas. Simulamos dos estrategias:
1. **Recomendación por popularidad**: recomendar las películas más valoradas.
2. **Recomendación basada en metadata**: si tenemos géneros, podemos recomendar según preferencias iniciales.

### 4.1. Estrategia 1: Recomendación por popularidad

In [ ]:
# Calculamos las películas más populares (mayor número de ratings y mejor promedio)
movie_stats = df_ratings.groupby('item_id').agg(
    count=('rating', 'count'),
    mean_rating=('rating', 'mean')
).reset_index()

# Filtramos películas con al menos 10 ratings
popular_movies = movie_stats[movie_stats['count'] >= 10].sort_values(['mean_rating', 'count'], ascending=False).head(10)
popular_movies = popular_movies.merge(df_movies[['movie_id', 'title']], left_on='item_id', right_on='movie_id')

print("Recomendaciones para un nuevo usuario (basadas en popularidad):")
for _, row in popular_movies.iterrows():
    print(f"- {row['title']} (promedio: {row['mean_rating']:.2f}, {row['count']} ratings)")

### 4.2. Estrategia 2: Recomendación basada en metadata (géneros)

Simulamos que el nuevo usuario indica preferencia por ciertos géneros.

In [ ]:
# Preferencias del nuevo usuario (simuladas)
user_genre_preferences = {
    'action': 5,
    'comedy': 3,
    'drama': 1,
    'horror': 0,
    'romance': 2
}

# Creamos una puntuación para cada película basada en géneros
genre_columns = ['unknown', 'action', 'adventure', 'animation', 'childrens', 'comedy', 
                 'crime', 'documentary', 'drama', 'fantasy', 'film_noir', 'horror', 
                 'musical', 'mystery', 'romance', 'sci_fi', 'thriller', 'war', 'western']

def compute_genre_score(row, preferences):
    score = 0
    for genre, pref in preferences.items():
        if genre in genre_columns and row[genre] == 1:
            score += pref
    return score

# Aplicamos a df_movies
df_movies['genre_score'] = df_movies.apply(lambda row: compute_genre_score(row, user_genre_preferences), axis=1)

# Recomendamos las películas con mayor puntuación de géneros
top_genre_movies = df_movies.sort_values('genre_score', ascending=False).head(10)

print("Recomendaciones para un nuevo usuario (basadas en preferencias de géneros):")
for _, row in top_genre_movies.iterrows():
    print(f"- {row['title']} (score géneros: {row['genre_score']})")

### 4.3. Combinación de popularidad y metadata

Podemos combinar ambas estrategias para un cold start más efectivo.

In [ ]:
# Normalizamos las puntuaciones
max_pop = popular_movies['mean_rating'].max()
max_genre = df_movies['genre_score'].max()

df_movies['popularity_score'] = df_movies['movie_id'].map(
    movie_stats.set_index('item_id')['mean_rating']
).fillna(0)

df_movies['combined_score'] = (df_movies['genre_score'] / max_genre * 0.7 + 
                                 df_movies['popularity_score'] / max_pop * 0.3)

top_combined = df_movies.sort_values('combined_score', ascending=False).head(10)

print("Recomendaciones combinadas (70% género, 30% popularidad):")
for _, row in top_combined.iterrows():
    print(f"- {row['title']} (score combinado: {row['combined_score']:.3f})")

---
## 5. Conclusiones

En este notebook hemos aplicado diferentes técnicas de sistemas de recomendación:

✔️ **Exploración de datos**: Analizamos la distribución de ratings, usuarios y películas.

✔️ **Filtrado colaborativo basado en ítems**:
- Calculamos similitud de coseno entre ítems.
- Implementamos una función de recomendación.
- Evaluamos con RMSE en una muestra.

✔️ **SVD con Surprise**:
- Entrenamos SVD y evaluamos con validación cruzada.
- Obtuvimos métricas RMSE y MAE.

✔️ **Cold start**:
- Simulamos estrategias basadas en popularidad.
- Usamos metadata de géneros para personalizar.
- Combinamos ambas para una solución híbrida.

**Lección clave**: Los sistemas de recomendación requieren combinar múltiples estrategias según el contexto. El cold start es un desafío real que se aborda con datos auxiliares (metadata) y reglas de negocio.

---
**Fin del Notebook de Ejercicios - Semana 12**